
***

### **What is a schema?**

*   In **Unity Catalog**, a schema is the **second level** in the namespace hierarchy:
        catalog.schema.table
*   It is a **child of a catalog** and can contain:
    *   **Tables**
    *   **Views**
    *   **Volumes**
    *   **Models**
    *   **Functions**

Schemas provide **granular organization** of data and AI assets, typically by:

*   **Use case**
*   **Project**
*   **Team sandbox**

***

### **Why use schemas?**

*   **Logical grouping**: More specific than catalogs.
*   **Access control**: Easier to manage permissions at schema level.
*   **Discoverability**: Helps users find relevant data quickly.

***

### **Terminology note**

*   In Azure Databricks, **schema = database**.
    *   `CREATE DATABASE` is an alias for `CREATE SCHEMA`.
*   This differs from traditional RDBMS where a database contains multiple schemas.

***

### **Managed storage locations**

*   You can **physically isolate data** for managed tables and volumes in a schema by specifying a **managed storage location**.
*   If not specified:
    *   Data defaults to the catalog’s managed location.
    *   If catalog has none, it falls back to the metastore’s managed location.
*   External tables and volumes: Isolation depends on **cloud storage management**, not schema settings.

***

✅ **Key takeaway**:  
Schemas in Unity Catalog are **organizational containers** under catalogs, used for finer control and logical grouping of data assets.

***


.



***

### ✅ **Before you begin**

To create a schema in **Unity Catalog**, you need:

*   A **Unity Catalog metastore** linked to your workspace.
*   **Permissions**:
    *   `USE CATALOG` and `CREATE SCHEMA` on the parent catalog.
    *   If specifying a managed storage location, `CREATE MANAGED STORAGE` on the external location.
*   A **Unity Catalog-compliant cluster** (standard/dedicated access mode) or a **SQL warehouse**.
*   For Hive metastore: permissions depend on table access control.

***

### **Ways to create a schema**

You can use:

1.  **Catalog Explorer (UI)**
2.  **SQL commands**
3.  **Terraform provider** (`databricks_schema`)

***

#### **Option 1: Catalog Explorer**

1.  Log in to a workspace linked to Unity Catalog.
2.  Click **Data → Catalog**.
3.  Select the **catalog** where you want the schema.
4.  Click **Create schema**.
5.  Enter:
    *   **Schema name**
    *   (Optional) **Comment**
    *   (Optional) **Managed storage location** (requires privilege).
6.  Click **Create**.
7.  Grant privileges on the schema (see *Manage privileges in Unity Catalog*).

***

#### **Option 2: SQL**

Run in a notebook or SQL editor:

```sql
CREATE { DATABASE | SCHEMA } [ IF NOT EXISTS ] <catalog-name>.<schema-name>
    [ MANAGED LOCATION '<location-path>' | LOCATION '<location-path>' ]
    [ COMMENT '<comment>' ]
    [ WITH DBPROPERTIES ( <property-key = property-value [, ...]> ) ];
```

**Parameters:**

*   `<catalog-name>`: Parent catalog (use `hive_metastore` for Hive metastore).
*   `<schema-name>`: Schema name.
*   `<location-path>`: Optional managed storage path.
*   `<comment>`: Optional description.
*   `DBPROPERTIES`: Optional Spark SQL properties.

***

#### **Option 3: Terraform**

*   Use `databricks_schema` resource.
*   Retrieve schema IDs with `databricks_schemas`.

***

### **After creation**

*   Grant privileges on the schema for users/groups.
*   If using managed storage, ensure external location permissions are correct.

***

✅ **Key takeaway**:  
Schemas can be created via UI, SQL, or Terraform. For Unity Catalog, permissions and managed storage options are critical. For Hive metastore, use SQL only.

***



.




***

### ✅ **1. Catalog Explorer (UI)**

**Steps:**

1.  Log in to your Databricks workspace linked to Unity Catalog.
2.  Click **Data → Catalog**.
3.  Select the catalog (e.g., `finance_catalog`).
4.  Click **Create schema**.
5.  Fill in:
    *   **Name**: `sales_reporting`
    *   **Comment**: `Schema for sales analytics and reporting`
    *   **Managed Location** (optional): `abfss://storage@account.dfs.core.windows.net/sales`
6.  Click **Create**.
7.  Grant privileges (e.g., `GRANT USE SCHEMA ON finance_catalog.sales_reporting TO analyst_group`).

***

### ✅ **2. SQL Command**

Run in a notebook or SQL editor:

**Unity Catalog example:**

```sql
CREATE SCHEMA IF NOT EXISTS finance_catalog.sales_reporting
    MANAGED LOCATION 'abfss://storage@account.dfs.core.windows.net/sales'
    COMMENT 'Schema for sales analytics and reporting'
    WITH DBPROPERTIES ('owner'='data_team', 'environment'='production');
```

**Hive Metastore example:**

```sql
CREATE DATABASE IF NOT EXISTS sales_reporting
    LOCATION 'abfss://storage@account.dfs.core.windows.net/hive/sales'
    COMMENT 'Legacy Hive schema for sales data';
```

***

### ✅ **3. Terraform**

Use the **Databricks Terraform provider**:

```hcl
resource "databricks_schema" "sales_reporting" {
  catalog_name = "finance_catalog"
  name         = "sales_reporting"
  comment      = "Schema for sales analytics and reporting"
  properties = {
    owner       = "data_team"
    environment = "production"
  }
  storage_root = "abfss://storage@account.dfs.core.windows.net/sales"
}
```

To list schemas:

```hcl
data "databricks_schemas" "all" {
  catalog_name = "finance_catalog"
}
```

***

✅ These examples cover **UI**, **SQL**, and **Terraform** methods for both Unity Catalog and Hive metastore.

***


.



***

## ✅ **Overview**

Managing schemas in **Unity Catalog** involves:

*   **Viewing schemas** and their details.
*   **Updating schemas** (owner, tags, comments, privileges).
*   **Deleting schemas** safely.

> **Note:** In the **legacy Hive metastore**, you must use SQL commands, and permissions differ based on table access control.

***

## ✅ **Before You Begin**

To manage schemas in Unity Catalog:

*   Your workspace must be linked to a **Unity Catalog metastore**.
*   Use a **Unity Catalog-compliant cluster** (standard/dedicated access mode) or a **SQL warehouse**.
*   Permissions vary by action:
    *   **View**: `USE SCHEMA` + `USE CATALOG` on parent catalog.
    *   **Update**: Depends on the type of update (explained below).
    *   **Delete**: Must be the **schema owner**.

***

## ✅ **Find and View Schemas**

### **Permissions**

*   `USE SCHEMA` on the schema + `USE CATALOG` on the parent catalog.
*   To view tables/views inside the schema: `SELECT` on those objects.

### **Methods**

#### **Catalog Explorer (UI)**

1.  Log in to a workspace linked to Unity Catalog.
2.  Click **Data → Catalog**.
3.  Select the catalog (e.g., `finance_catalog`) or search using **Type to filter**.
4.  Click the schema name to open details.

#### **SQL Commands**

*   **List schemas in a catalog**:

```sql
SHOW SCHEMAS IN finance_catalog;
```

*   **Filter by pattern**:

```sql
SHOW SCHEMAS IN finance_catalog LIKE 'sales_*';
```

*   **Describe schema details**:

```sql
DESCRIBE SCHEMA finance_catalog.sales_reporting;
```

***

## ✅ **Update (Alter) a Schema**

### **Permissions Required**

*   **Change owner**: Owner OR `MANAGE + USE SCHEMA` + `USE CATALOG`.
*   **Rename schema**: Owner OR same as above.
*   **Add/update comment**: Owner OR `MANAGE + USE SCHEMA` + `USE CATALOG`.
*   **Add/update tags**: Owner OR `MODIFY + USE SCHEMA` + `USE CATALOG`.
*   **Add table**: Owner OR `CREATE TABLE + USE SCHEMA` + `USE CATALOG`.
*   **Add volume**: Owner OR `CREATE VOLUME + USE SCHEMA` + `USE CATALOG`.
*   **Grant/revoke permissions**: Owner OR metastore admin OR `MANAGE + USE SCHEMA` + `USE CATALOG`.

***

### **Catalog Explorer (UI)**

On the schema details page:

*   **Overview tab**:
    *   Change **owner**.
    *   Add/update **tags**.
    *   Add/update **comments**.
*   **Permissions tab**:
    *   Grant/revoke privileges.
*   **Kebab menu**:
    *   Rename schema.
*   **Create button**:
    *   Add tables or volumes.

***

### **SQL Examples**

*   **Change owner**:

```sql
ALTER SCHEMA finance_catalog.sales_reporting OWNER TO new_owner;
```

*   **Add/update comment**:

```sql
ALTER SCHEMA finance_catalog.sales_reporting
SET COMMENT 'Updated schema for sales analytics';
```

*   **Add/update tags**:

```sql
ALTER SCHEMA finance_catalog.sales_reporting
SET TAGS ('environment'='production', 'owner'='data_team');
```

*   **Grant privileges**:

```sql
GRANT USE SCHEMA ON SCHEMA finance_catalog.sales_reporting TO analyst_group;
```

*   **Revoke privileges**:

```sql
REVOKE USE SCHEMA ON SCHEMA finance_catalog.sales_reporting FROM analyst_group;
```

> **Important:** SQL does **not** support direct rename. To rename, create a new schema and move assets manually.

***

## ✅ **Delete (Drop) a Schema**

### **Permissions**

*   Must be the **schema owner**.

### **Catalog Explorer (UI)**

1.  Delete all tables in the schema first.
2.  Go to **Data → Catalog → Schema**.
3.  Click **Kebab menu → Delete → Confirm**.

### **SQL Examples**

*   **Drop schema only if empty**:

```sql
DROP SCHEMA finance_catalog.sales_reporting RESTRICT;
```

*   **Drop schema and all its objects**:

```sql
DROP SCHEMA finance_catalog.sales_reporting CASCADE;
```

*   **Safe delete if exists**:

```sql
DROP SCHEMA IF EXISTS finance_catalog.sales_reporting CASCADE;
```

***

## ✅ **Key Notes**

*   **CASCADE** deletes all tables and objects in the schema.
*   **RESTRICT** fails if schema contains objects.
*   For **Hive metastore**, omit catalog name:

```sql
DROP SCHEMA sales_reporting CASCADE;
```

***

### 🔍 **Summary Table**

| Action       | Method | Example Command                   |
| ------------ | ------ | --------------------------------- |
| View schemas | SQL    | `SHOW SCHEMAS IN catalog;`        |
| Describe     | SQL    | `DESCRIBE SCHEMA catalog.schema;` |
| Update       | SQL    | `ALTER SCHEMA ...`                |
| Delete       | SQL    | `DROP SCHEMA ... CASCADE`         |

***

✅ This is the **full detailed guide** for managing schemas in Unity Catalog.

***
